In [5]:
# GUI/app.py
import sys
import os
import threading
import speech_recognition as sr
from flask import Flask, render_template, request, jsonify
from flask_socketio import SocketIO
import nest_asyncio

# Add parent folder to path (works in Jupyter)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from pi_client import send_command_to_pi
from voice_input import listen_command

# Allow Flask to run in Jupyter
nest_asyncio.apply()

# Set current folder as base for templates and static
BASE_DIR = os.getcwd()
app = Flask(__name__, static_folder=os.path.join(BASE_DIR, "static"),
            template_folder=os.path.join(BASE_DIR, "templates"))

socketio = SocketIO(app, cors_allowed_origins="*", async_mode='threading')

# Home page
@app.route('/')
def index():
    return render_template('index.html')


# Endpoint to send command to robot
@app.route('/send', methods=['POST'])
def send():
    data = request.json
    command = data.get('command')
    socketio.emit('status', {'state': 'thinking', 'message': '🤔 Dora is thinking...'})
    threading.Thread(target=send_and_update, args=(command,)).start()
    return jsonify({'ok': True})


# Voice input endpoint
@app.route('/voice', methods=['GET'])
def voice_command():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("🎤 Listening for voice input...")
        audio = recognizer.listen(source)
        try:
            command = recognizer.recognize_google(audio)
            print(f"✅ Recognized: {command}")
            return jsonify({"command": command})
        except sr.UnknownValueError:
            print("❌ Could not understand audio")
            return jsonify({"command": ""})
        except sr.RequestError as e:
            print(f"⚠️ Speech recognition service error: {e}")
            return jsonify({"command": ""})

# Function to handle sending command and updating GUI
def send_and_update(command):
    try:
        socketio.emit('status', {'state': 'thinking', 'message': '🤔 Dora is thinking...'})
        response = send_command_to_pi(command)
        state = response.get('status', 'sad')
        message = response.get('message', '')

        # Friendly child-like messages
        if 'Comm error' in message:
            socketio.emit('status', {'state': 'sad', 'message': "Oops! I couldn’t find that item!"})
        else:
            socketio.emit('status', {'state': state, 'message': message})
    except Exception as e:
        socketio.emit('status', {'state': 'sad', 'message': "Uh-oh! Something went wrong."})

socketio.run(app, host='0.0.0.0', port=5000, debug=True, use_reloader=False, allow_unsafe_werkzeug=True)

Werkzeug appears to be used in a production deployment. Consider switching to a production web server instead.


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.230.133.63:5000
Press CTRL+C to quit
10.230.133.63 - - [26/Nov/2025 16:25:16] "GET / HTTP/1.1" 200 -
10.230.133.63 - - [26/Nov/2025 16:25:16] "GET /static/dora_idle.png HTTP/1.1" 304 -
10.230.133.63 - - [26/Nov/2025 16:25:17] "GET /static/forest_bg.jpg HTTP/1.1" 304 -
10.230.133.63 - - [26/Nov/2025 16:25:17] "GET /static/bg.wav HTTP/1.1" 206 -
10.230.133.63 - - [26/Nov/2025 16:25:17] "GET /static/bg.wav HTTP/1.1" 206 -
10.230.133.63 - - [26/Nov/2025 16:25:17] "GET /static/suspense.wav HTTP/1.1" 206 -
10.230.133.63 - - [26/Nov/2025 16:25:17] "GET /static/bg.wav HTTP/1.1" 206 -
10.230.133.63 - - [26/Nov/2025 16:25:17] "GET /static/happy.wav HTTP/1.1" 206 -
10.230.133.63 - - [26/Nov/2025 16:25:17] "GET /socket.io/?EIO=4&transport=polling&t=Pg_H5CZ HTTP/1.1" 200 -
10.230.133.63 - - [26/Nov/2025 16:25:17] "POST /socket.io/?EIO=4&transport=polling&t=Pg_H5F9&sid=gwQLYQD2ZTDMjTHsAAAA HTTP/1.1" 200

In [2]:
!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.8/32.8 MB 6.0 MB/s eta 0:00:00:00:0100:01

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
!pip install flask-socketio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 230.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 827.5 kB/s eta 0:00:000:00:01

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
!pip install pipwin
!pipwin install pyaudio

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.1/74.1 kB 1.2 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.2 MB/s eta 0:00:0000:0100:01
  Created wheel for pipwin: filename=pipwin-0.5.2-py2.py3-none-any.whl size=9938 sha256=f85b454d5cc21692f2515d15aff0927043e5f89b9b6dd16823adc07dcbaa8332
  Stored in directory: /root/.cache/pip/wheels/f4/a8/76/372879a40d2d4dd7de23efd867ed37112687429b8d0dda4545
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=19867 sha256=cd0d7e8e36393e60fd2fed77c4c406fa87a42fd1718fcf0875024c67c152c84f
  Stored in directory: /root/.cache/pip/wheels/56/ea/58/ead137b087d9e326852a851351d1debf4ada529b6ac0ec4e8c
  Created wheel for pyjsparser: filename=pyjsp

In [2]:
import sys
print(sys.executable)
import importlib.util
print(importlib.util.find_spec("pyaudio"))



/usr/bin/python3
None
